# Audiobooks Preprocessing (Aligned with `src/preprocess.py`)

This notebook mirrors the production preprocessing pipeline exactly:
- load raw CSV
- stratified train/validation/test split
- fit scaler on train only (no leakage)
- transform validation/test
- save artifacts to `artifacts/data/`

In [ ]:
from pathlib import Path
import joblib
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [ ]:
SEED = 42
TEST_SIZE = 0.1
VALIDATION_SIZE = 0.1

INPUT_CSV = Path("Audiobooks_data.csv")
OUT_DIR = Path("artifacts/data")
OUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
raw = np.loadtxt(INPUT_CSV, delimiter=",")
inputs = raw[:, 1:-1]
targets = raw[:, -1].astype(np.int32)

val_ratio_adjusted = VALIDATION_SIZE / (1.0 - TEST_SIZE)

x_train_val, x_test, y_train_val, y_test = train_test_split(
    inputs,
    targets,
    test_size=TEST_SIZE,
    random_state=SEED,
    stratify=targets,
)

x_train, x_val, y_train, y_val = train_test_split(
    x_train_val,
    y_train_val,
    test_size=val_ratio_adjusted,
    random_state=SEED,
    stratify=y_train_val,
)

In [ ]:
scaler = StandardScaler()
x_train_scaled = scaler.fit_transform(x_train).astype(np.float32)
x_val_scaled = scaler.transform(x_val).astype(np.float32)
x_test_scaled = scaler.transform(x_test).astype(np.float32)

np.savez(OUT_DIR / "train.npz", inputs=x_train_scaled, targets=y_train)
np.savez(OUT_DIR / "validation.npz", inputs=x_val_scaled, targets=y_val)
np.savez(OUT_DIR / "test.npz", inputs=x_test_scaled, targets=y_test)
joblib.dump(scaler, OUT_DIR / "scaler.joblib")

In [ ]:
print(f"Raw samples: {len(targets)}, positive rate: {targets.mean():.4f}")
print(f"Train: {x_train_scaled.shape}, positives: {int(y_train.sum())}/{len(y_train)}")
print(f"Val:   {x_val_scaled.shape}, positives: {int(y_val.sum())}/{len(y_val)}")
print(f"Test:  {x_test_scaled.shape}, positives: {int(y_test.sum())}/{len(y_test)}")
print(f"Saved artifacts to: {OUT_DIR.resolve()}")